# Module 15: Interactive SQLAlchemy 2.0 & Async Database Architecture

### What You Will Discover
By running this notebook, you will explore SQLAlchemy 2.0's typed declarative models (`Mapped[]`, `mapped_column()`), execute async queries with explicit `select()`, and eliminate the $N+1$ query problem with `selectinload`.

**Key Question Answered:** *Why does accessing related attributes outside an active async session raise `DetachedInstanceError`, and how does eager loading fix it?*


In [ ]:
# Step 1: Defining 2.0 DeclarativeBase and Models

from sqlalchemy import ForeignKey, String
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship


class Base(DeclarativeBase):
    pass

class Author(Base):
    __tablename__ = 'authors'
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(50))
    books: Mapped[list['Book']] = relationship(back_populates='author')

class Book(Base):
    __tablename__ = 'books'
    id: Mapped[int] = mapped_column(primary_key=True)
    title: Mapped[str] = mapped_column(String(100))
    author_id: Mapped[int] = mapped_column(ForeignKey('authors.id'))
    author: Mapped[Author] = relationship(back_populates='books')


In [ ]:
# Step 2: Creating an in-memory SQLite async engine and tables
import asyncio

from sqlalchemy.ext.asyncio import async_sessionmaker, create_async_engine

engine = create_async_engine('sqlite+aiosqlite:///:memory:', echo=False)
async with engine.begin() as conn:
    await conn.run_sync(Base.metadata.create_all)

SessionLocal = async_sessionmaker(engine, expire_on_commit=False)
print('Async engine and SQLite schema initialized.')


In [ ]:
# Step 3: Inserting sample relational data
async with SessionLocal() as session, session.begin():
    a = Author(name='Ada Lovelace')
    b1 = Book(title='Notes on the Analytical Engine', author=a)
    session.add(a)
print('Author and Book records committed.')


### 🔮 Prediction Prompt
**Before running the next cell:** If we query `Author` without `selectinload(Author.books)` and then access `author.books` in async code, will SQLAlchemy emit a query automatically, or will it raise an error?


In [ ]:
# Surprising Result: Implicit Async I/O Is Forbidden in 2.0
from sqlalchemy import select

async with SessionLocal() as session:
    stmt = select(Author).where(Author.name == 'Ada Lovelace')
    res = await session.execute(stmt)
    author = res.scalar_one()
    try:
        # Accessing unloaded relation without await triggers MissingGreenlet / DetachedInstanceError
        _ = author.books
        print('Loaded books without error.')
    except Exception as exc:
        print(f'Caught expected error:\n{type(exc).__name__}: {exc}')
        print('Explanation: Async ORM prevents implicit synchronous database reads on attribute access!')


### The Solution: Eager Loading with `selectinload`
`selectinload` tells SQLAlchemy to fetch related records in an explicit second query using `WHERE author_id IN (...)`.


In [ ]:
from sqlalchemy.orm import selectinload

async with SessionLocal() as session:
    stmt = select(Author).options(selectinload(Author.books))
    res = await session.execute(stmt)
    author_eager = res.scalar_one()
    print(f'Author: {author_eager.name}')
    print(f'Books eagerly loaded: {[b.title for b in author_eager.books]}')


### Cleaning Up Async Engine Connections
Always dispose of the engine upon application shutdown to release connection pools.


In [ ]:
await engine.dispose()
print('Async connection pool cleanly disposed.')


### 🛠️ Interactive Challenge: Fix the Shared Session Anti-Pattern
The following function attempts to run two concurrent database queries sharing the exact same `AsyncSession` instance, causing concurrency corruption. Fix it so each concurrent worker gets its own session.


In [ ]:
# TODO: FIX ME - Give each concurrent worker its own session context
async def query_worker(worker_id: int):
    # FIX: async with SessionLocal() as session:
    # Do not pass a single shared session across multiple concurrent tasks!
    return f'Worker {worker_id} executed with isolated session'

results = await asyncio.gather(query_worker(1), query_worker(2))
print(f'Results: {results}')


### 🏁 Summary & Next Steps
- SQLAlchemy 2.0 uses explicit `select()` statements and typed `Mapped[]` columns.
- In async code, always use `options(selectinload(...))` to avoid $N+1$ query storms.
- Run `python 01_declarative_models_demo.py` and `python 02_async_sessions_and_relations_demo.py`.
- Follow [PROJECT_GUIDE.md](PROJECT_GUIDE.md) to implement the ledger database.
